# 03 — LSTM Forecasting Model

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Load feature data

In [ ]:
featured = pd.read_csv('../data/processed/features.csv', parse_dates=['date'])
from src.features import get_feature_columns
FEATURE_COLS = [c for c in get_feature_columns() if c in featured.columns]
TARGET = 'new_cases_7d'
print(f"Features: {len(FEATURE_COLS)}")

## 2. Prepare per-country sequences

In [ ]:
from src.models.lstm_model import LSTMForecaster

# Train on one country first
COUNTRY = 'United States'
cdf = featured[featured.country == COUNTRY].sort_values('date').reset_index(drop=True)

cutoff = cdf['date'].max() - pd.Timedelta(days=30)
train = cdf[cdf.date <= cutoff]
test  = cdf[cdf.date >  cutoff]

X_train = train[FEATURE_COLS].fillna(0).values
y_train = train[TARGET].fillna(0).values
X_test  = test[FEATURE_COLS].fillna(0).values
y_test  = test[TARGET].fillna(0).values

print(f"Train: {X_train.shape}  Test: {X_test.shape}")

## 3. Train LSTM

In [ ]:
model = LSTMForecaster(seq_len=21, units=64, epochs=30)
model.fit(X_train, y_train, validation_split=0.1)

## 4. Evaluate

In [ ]:
from src.evaluation import evaluate, plot_forecast

preds = model.predict(X_test)
metrics = evaluate(y_test[:len(preds)], preds, name='LSTM')

plot_forecast(test['date'].values[:len(preds)],
              y_test[:len(preds)], preds,
              country=COUNTRY, model_name='LSTM')

## 5. Save model

In [ ]:
model.save('../models/saved/lstm.keras')